In [3]:
import json
import requests

response = requests.get("https://api.crossref.org/v1/works?filter=issn:0003-0554&sort=published&order=desc&rows=365&mailto=sam.kaenner@uni-konstanz.de").json()
articles = response["message"]["items"]

In [4]:
import re
import pandas as pd

def clean_abstract(text):
    return re.sub(r"<[^>]+>", "", text).strip()

results = []

for a in articles:
    results.append({
        "title": a.get("title", [""])[0],
        "doi": a.get("DOI"),
        "published": a.get("published", {}).get("date-parts", [[]])[0],
        "ref_count": a.get("reference-count"),
        "authors": a.get("author"),
        "abstract": clean_abstract(a.get("abstract", "No abstract available"))
    })

pd.DataFrame(results)

,title,doi,published,ref_count,authors,abstract
0,Elite Partisan Disagreement and Military Victo...,10.1017/s0003055426101543,"[2026, 3, 24]",161,[{'ORCID': 'https://orcid.org/0000-0001-8248-4...,Does partisan disagreement impact expectations...
1,The Effects of Exposure to New Electoral Rules...,10.1017/s0003055426101518,"[2026, 3, 23]",82,"[{'given': 'AVI', 'family': 'AHUJA', 'sequence...",How do electoral rules influence voters? We bu...
2,What Is the Wrong of Capitalism? A Reply to Ch...,10.1017/s0003055426101567,"[2026, 3, 23]",37,[{'ORCID': 'https://orcid.org/0000-0002-9819-3...,Chiara Cordelli has recently criticized radica...
3,Do Elites Know Best? Candidate Selection and P...,10.1017/s0003055426101531,"[2026, 3, 23]",124,[{'ORCID': 'https://orcid.org/0000-0003-2234-9...,How do candidate selection processes shape pol...
4,Exiting Russia,10.1017/s000305542610152x,"[2026, 3, 11]",92,[{'ORCID': 'https://orcid.org/0000-0002-7430-9...,"As of February 24, 2022, over forty thousand f..."
...,...,...,...,...,...,...
360,Female Representation and Legitimacy: Evidence...,10.1017/s0003055423000357,"[2023, 4, 27]",40,[{'ORCID': 'https://orcid.org/0000-0001-5566-0...,How does the gender composition of deliberativ...
361,"Extraction, Assimilation, and Accommodation: T...",10.1017/s0003055423000333,"[2023, 4, 26]",76,[{'ORCID': 'https://orcid.org/0000-0002-9472-1...,Why do some Indigenous communities experience ...
362,Fathers’ Leave Reduces Sexist Attitudes,10.1017/s0003055423000369,"[2023, 4, 26]",26,[{'ORCID': 'https://orcid.org/0000-0002-6443-1...,Research shows that sexist attitudes are deepl...
363,Contested Killings: The Mobilizing Effects of ...,10.1017/s0003055423000321,"[2023, 4, 26]",83,[{'ORCID': 'https://orcid.org/0000-0001-7725-6...,"Recently, we have witnessed the politicizing e..."


In [ ]:
results_df = pd.DataFrame(results)


import pandas as pd

results_df["author_names"] = results_df["authors"].apply(
    lambda t: (
        [
            f'{a.get("family", pd.NA)}, {a.get("given", pd.NA)}'
            for a in t
        ]
        if t is not None else pd.NA
    )
)


In [15]:
import time
import requests
import pandas as pd

HEADERS = {"User-Agent": "polisci-ish/0.1 (mailto:sam.kaenner@uni-konstanz.de)"}

def fetch_crossref_metadata(doi):
    if pd.isna(doi) or not str(doi).strip():
        return {}

    doi = str(doi).strip()
    url = f"https://api.crossref.org/works/{doi}"

    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
        message = r.json().get("message", {})
        return {
            "journal": (message.get("container-title") or [pd.NA])[0],
            "volume": message.get("volume", pd.NA),
            "issue": message.get("issue", pd.NA),
            "pages": message.get("page", pd.NA),
            "article_number": message.get("article-number", pd.NA),
            "publisher": message.get("publisher", pd.NA),
            "crossref_title": (message.get("title") or [pd.NA])[0],
        }
    except Exception as e:
        return {
            "journal": pd.NA,
            "volume": pd.NA,
            "issue": pd.NA,
            "pages": pd.NA,
            "article_number": pd.NA,
            "publisher": pd.NA,
            "crossref_title": pd.NA,
            "crossref_error": str(e),
        }
    
meta_rows = []

for doi in results_df["doi"]:
    meta_rows.append(fetch_crossref_metadata(doi))
    time.sleep(0.1)

crossref_df = pd.DataFrame(meta_rows)
results_df = pd.concat([results_df.reset_index(drop=True), crossref_df], axis=1)

def format_authors_apa(authors):
    if authors is None or (isinstance(authors, float) and pd.isna(authors)):
        return "[AUTHOR MISSING]"

    formatted = []
    for a in authors:
        given = a.get("given", "")
        family = a.get("family", "")

        initials = " ".join(f"{part[0]}." for part in str(given).split() if part)
        if family and initials:
            formatted.append(f"{family}, {initials}")
        elif family:
            formatted.append(str(family))
        elif initials:
            formatted.append(initials)
        else:
            formatted.append("[AUTHOR NAME MISSING]")

    if not formatted:
        return "[AUTHOR MISSING]"

    if len(formatted) == 1:
        return formatted[0]
    if len(formatted) <= 20:
        return ", ".join(formatted[:-1]) + ", & " + formatted[-1]

    # APA 7: 21+ authors → first 19, ellipsis, final author
    return ", ".join(formatted[:19]) + ", ... " + formatted[-1]

def get_year(parts):
    if parts is None:
        return "n.d."
    try:
        if isinstance(parts, str):
            import ast
            parts = ast.literal_eval(parts)
        if isinstance(parts, (list, tuple)) and len(parts) >= 1:
            return str(int(parts[0]))
    except Exception:
        pass
    return "n.d."

def is_missing(x):
    return x is None or pd.isna(x) or str(x).strip() == ""

def build_apa_citation(row):
    authors = format_authors_apa(row.get("authors"))
    year = get_year(row.get("published"))
    title = row.get("title", "[TITLE MISSING]")
    journal = row.get("journal")
    volume = row.get("volume")
    issue = row.get("issue")
    pages = row.get("pages")
    article_number = row.get("article_number")
    doi = row.get("doi")

    loc = pages
    if is_missing(loc) and not is_missing(article_number):
        loc = f"Article {article_number}"
    if is_missing(loc):
        loc = "[PAGES OR ARTICLE NUMBER MISSING]"

    journal = "[JOURNAL NAME MISSING]" if is_missing(journal) else journal
    volume = "[VOLUME MISSING]" if is_missing(volume) else volume
    issue_text = "" if is_missing(issue) else f"({issue})"
    doi_text = "" if is_missing(doi) else f" https://doi.org/{doi}"

    return f"{authors} ({year}). {title}. *{journal}, {volume}*{issue_text}, {loc}.{doi_text}"

In [ ]:
def build_apa_citation(row):
    authors = format_authors_apa(row.get("authors"))
    year = get_year(row.get("published"))
    title = row.get("title", "[TITLE MISSING]")
    journal = row.get("journal")
    volume = row.get("volume")
    issue = row.get("issue")
    pages = row.get("pages")
    article_number = row.get("article_number")
    doi = row.get("doi")

    loc = pages
    if is_missing(loc) and not is_missing(article_number):
        loc = f"Article {article_number}"
    if is_missing(loc):
        loc = "[PAGES OR ARTICLE NUMBER MISSING]"

    journal = "[JOURNAL NAME MISSING]" if is_missing(journal) else journal
    volume = "[VOLUME MISSING]" if is_missing(volume) else volume
    issue_text = "" if is_missing(issue) else f"({issue})"
    doi_text = "" if is_missing(doi) else f" https://doi.org/{doi}"

    return f"{authors.title()} ({year}). {title}. {journal}, {volume} {issue_text}, {loc}.[{doi_text}]({doi_text})"

results_df["apa_citation"] = results_df.apply(build_apa_citation, axis=1)
results_df.to_csv("data/clean_abstracts.csv")